# Live FPGA FFT over USB

Standalone copy of section 6 of `hw_fft_full.ipynb`: only needs `fft_live.py` in the same folder.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
os.makedirs('captures', exist_ok=True)


## 6. Live FFT over USB (`mic_fft_spi_top`)

With `mic_fft_spi_top` programmed, the FPGA sends every FFT frame on the SPI pins (J3: SCK P46, MOSI P45, CS P3) **and** the same frame over the UPduino's USB serial port, so this section reads it through the USB cable. Each frame is `A55A`, `{frame number, STAGE_SHIFT}`, then re/im of X[0..255], then a checksum word (`fft_live.py`, `bins_uart.sv`).

Before running:
- `pip install pyserial numpy matplotlib`
- after programming with openFPGALoader, **unplug and replug the USB cable** (or run `ftdi_reset.py` on Windows) so the FT232H is back in serial mode
- close any other program that has the serial port open

`PORT = None` finds the UPduino; otherwise set it, e.g. `'COM5'`, `'/dev/ttyUSB0'` or `'/dev/cu.usbserial-...'`.

In [ ]:
import time
from IPython.display import clear_output
import fft_live

PORT = None          # None = find the UPduino's FTDI port
SECONDS = 20         # how long to run the live plot
FS_LIVE = fft_live.FS
freqs_live = np.arange(fft_live.BINS) * FS_LIVE / 512

ser = fft_live.open_port(PORT)
print('reading', ser.port)
hold = np.full(fft_live.BINS, -100.0)
t0, n, last = time.time(), 0, 0.0
try:
    for f in fft_live.frames_from(ser):
        db = fft_live.magnitude_db(f)
        hold = np.maximum(db, hold - 0.5)
        n += 1
        if time.time() - last > 0.2 and ser.in_waiting < 2060:     # redraw about 5 times a second
            last = time.time()
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4.5))
            ax.plot(freqs_live, db, lw=1, label='this frame')
            ax.plot(freqs_live, hold, lw=0.8, alpha=0.5, label='peak hold')
            k = int(np.argmax(db[1:])) + 1
            ax.set_title(f'frame {f["frame"]}: peak {k * FS_LIVE / 512:.0f} Hz (bin {k}), {n / (time.time() - t0):.1f} frames/s')
            ax.set_xlim(0, FS_LIVE / 2); ax.set_ylim(-100, 0)
            ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('|X[k]| (dB, 0 = full-scale sine)')
            ax.grid(alpha=0.3); ax.legend(loc='upper right')
            plt.show()
        if time.time() - t0 > SECONDS:
            break
finally:
    ser.close()
print(f'{n} frames in {time.time() - t0:.1f} s')

### Record a few seconds and look at them

Records raw USB bytes to `captures/spi_NAME.bin`, then plots a spectrogram of every frame. The recordings can be re-plotted later without the board.

In [ ]:
NAME = 'live'
REC_SECONDS = 4

out = f'captures/spi_{NAME}.bin'
fft_live.record(PORT, REC_SECONDS, out)
rec, _, bad = fft_live.parse(open(out, 'rb').read())
nums = [f['frame'] for f in rec]
gaps = sum((b - a) % 256 != 1 for a, b in zip(nums, nums[1:]))
print(f'{len(rec)} frames, {bad} bad checksums, {gaps} frame-number gaps')

S = np.array([fft_live.magnitude_db(f) for f in rec])
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(S.T, origin='lower', aspect='auto', vmin=-90, vmax=-10, cmap='magma',
               extent=[0, len(rec) * 312 / FS_LIVE, 0, FS_LIVE / 2])
ax.set_xlabel('time (s)'); ax.set_ylabel('frequency (Hz)')
ax.set_title(f'{out}: FPGA FFT read over USB')
fig.colorbar(im, label='dB')
plt.show()